## Demo notebook

In this demo we will go through the basic functionalities of `RotOptSynth`.

In [1]:
import numpy as np
from scipy.stats import unitary_group
import rotoptsynth as ros
import pennylane as qml

In [2]:
# Pick some system size and generate a target with that size.
n = 4
n = 7
N = 2**n
wires = range(n)
dim = 4**n
U = unitary_group.rvs(N, random_state=81512)

In [6]:
# Perform unitary synthesis with validation enabled.
ros.disable_validation()
%prun -s tottime ops = ros.rot_opt_qsd(U, wires=wires, zeroed_wires=None)

         2556875 function calls (2522984 primitive calls) in 0.671 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    55569    0.040    0.000    0.096    0.000 wires.py:42(_process)
57324/50125    0.029    0.000    0.099    0.000 autoray.py:30(do)
    20543    0.025    0.000    0.087    0.000 operation.py:1099(__init__)
     1023    0.018    0.000    0.082    0.000 asym_decomp.py:220(asymmetric_two_qubit_decomp)
    62849    0.017    0.000    0.028    0.000 interface_utils.py:151(_get_interface_of_single_tensor)
      677    0.017    0.000    0.200    0.000 flag_decomp.py:140(attach_multiplexer_node)
197249/197245    0.017    0.000    0.026    0.000 {built-in method builtins.isinstance}
293084/267754    0.017    0.000    0.019    0.000 {built-in method builtins.len}
      341    0.014    0.000    0.037    0.000 _decomp_cossin.py:145(_cossin)
     2046    0.014    0.000    0.168    0.000 decomposition.py:30(zyz_rotation_ang

In [12]:
# Additionally check manually that the matrix is reproduced correctly:
print(f"Matrix reproduced correctly: {np.allclose(ros.ops_to_mat(ops, wires), U)}")

Matrix reproduced correctly: True


In [4]:
# Count the number of rotation angles in the operators and compare to the group dimension
num_rotation_angles = ros.count_rotation_angles(ops)
print(f"Number of rotation angles ({num_rotation_angles}) matches group dimension ({dim}): {num_rotation_angles==dim}")

Number of rotation angles (16384) matches group dimension (16384): True


In [5]:
# Draw the circuit
print(qml.drawer.tape_text(qml.tape.QuantumScript(ops), show_matrices=False, max_length=100, wire_order=wires))

0: ────────────────────────────────╭GlobalPhase───────────────────────────────────────────────── ···
1: ────────────────────────────────├GlobalPhase───────────────────────────────────────────────── ···
2: ────────────────────────────────├GlobalPhase───────────────────────────────────────────────── ···
3: ────────────────────────────────├GlobalPhase───────────────────────────────────────────────── ···
4: ────────────────────────────────├GlobalPhase─╭RZ(M4)──────────────────────────╭RY(M7)─╭RZ(M8) ···
5: ──U(M0)─╭X──RZ─╭●─────╭X──U(M2)─├GlobalPhase─├◑───────RY──RZ─╭●──RX─╭●──U(M5)─├◑──────├◑───── ···
6: ──U(M1)─╰●──RY─╰X──RY─╰●──U(M3)─╰GlobalPhase─╰◑───────RY──RZ─╰X──RZ─╰X──U(M6)─╰◑──────╰◑───── ···

0: ··· ─────────────────────────────────────────────────────────────────────────── ···
1: ··· ─────────────────────────────────────────────────────────────────────────── ···
2: ··· ─────────────────────────────────────────────────────────────────────────── ···
3: ··· ────────────────────────

In [6]:
# Count gates:
@qml.specs
@qml.qnode(qml.device("default.qubit"))
def node():
    for op in ops:
        qml.apply(op)

    return qml.expval(qml.Z(0))
    
resources = node()["resources"]
gate_counts = dict(resources.gate_types)
gate_counts

{'QubitUnitary': 6,
 'CNOT': 127,
 'RZ': 4,
 'RY': 4,
 'GlobalPhase': 1,
 'SelectPauliRot': 485,
 'RX': 1,
 'SelectSU2': 122}

In [7]:
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(ros.asymmetric_two_qubit_decomp, show_matrices=False)(U, wires=wires))

TypeError: asymmetric_two_qubit_decomp() got an unexpected keyword argument 'wires'

In [ ]:
from rotoptsynth.flag_decomp import _flag_decomp_two_qubits
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(_flag_decomp_two_qubits, show_matrices=False)(U, wires=wires))

In [ ]:
ros.attach_multiplexer_node([qml.RY(0.6, 0)], [qml.RY(-0.1, 0)], "mpx")

In [ ]:
ops0 = [qml.SelectPauliRot([0.6, 0.7], [0], target_wire="target", rot_axis="X")]
ops1 = [qml.SelectPauliRot([0.2, -0.5], [0], target_wire="target", rot_axis="X")]
ros.attach_multiplexer_node(ops0, ops1, "new mpx")

In [ ]:
ros.attach_multiplexer_node([qml.CNOT([0, 1])], [qml.CNOT([0, 1])], "mpx")